# 03 — Iterative Debugging Loop & Execution Reward Design
**Goal**: Implement the multi-turn agentic debugging loop ($K=3$ turns) with error feedback (Verification Check V2), define execution-guided reward functions, synthetic bug injection, and collect trajectory preference pairs for DPO.

---

## Step 1: Environment & Module Setup

In [ ]:
import sys, os

# Universal Path Resolution for Kaggle / Colab / Molab / Local
def find_repo_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.isdir(os.path.join(curr, 'src')):
            return curr
        curr = os.path.dirname(curr)
    for fallback in ['/kaggle/working/self-correction-llm-rl', '/kaggle/working', '/content/self-correction-llm-rl', '/content']:
        if os.path.isdir(os.path.join(fallback, 'src')):
            return fallback
    return os.path.abspath('..')

repo_root = find_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project root added to sys.path: {repo_root}")

from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import build_prompt, agentic_debug_loop, DebugLoop
from src.rewards.execution_reward import compute_reward, compute_reward_binary, compute_partial_reward, compute_status_aware_reward
from src.training.dpo import make_preference_pairs
from src.error_injection import BugInjector, BugType

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Modules imported successfully!")

## Step 2: Synthetic Bug Injection Demo

In [ ]:
injector = BugInjector(seed=42)
correct_code = "def multiply(a, b):\n    return a * b"

print("=== Synthetic Bug Injection ===")
for category in list(BugType):
    res = injector.inject_bug(correct_code, category=category)
    print(f"[{res['bug_type'].upper()}] -> {res['description']}")

## Step 3: Multi-Turn Agentic Debugging Loop (Verification Check V2)
Demonstrates the agentic debugging loop up to $K=3$ turns where model inspects its previous attempt and error traceback to fix bugs.

In [ ]:
model, tokenizer = load_model_and_tokenizer(load_in_4bit=True)

apps = load_dataset('codeparrot/apps', split='train[:10]', trust_remote_code=True)
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)

problem = apps_clean[0]['question']
print("--- APPS Problem Statement ---")
print(problem[:250] + "...")

print("\n--- Running Agentic Debugging Loop (K=3) ---")
history = agentic_debug_loop(model, tokenizer, problem, test_cases=[], K=3)

for turn_info in history:
    print(f"\n[Turn {turn_info['turn']}] Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:150]}...")
    if turn_info['result']['traceback']:
        print(f"Traceback: {turn_info['result']['traceback'][:150]}...")

print("\nVerification Check V2 completed!")

## Step 4: Execution Reward Function Verification
Validates reward scores for AC ($+1.0$), WA/partial pass ($0.0 - 1.0$), and CE/RE ($-0.2$ penalty).

In [ ]:
print("--- Dense Reward Calculations ---")
print(f"AC (5/5 tests): {compute_reward('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward('WA', 3, 5)}")
print(f"CE (0/5 tests): {compute_reward('CE', 0, 5)}")
print(f"RE (0/5 tests): {compute_reward('RE', 0, 5)}")

print("\n--- Binary Reward Calculations (RQ4 Ablation) ---")
print(f"AC (5/5 tests): {compute_reward_binary('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward_binary('WA', 3, 5)}")

## Step 5: Preference Pair Generation for DPO
Collects `(prompt, chosen, rejected)` trajectory pairs from debug rollouts for DPO training.

In [ ]:
print("Generating sample DPO preference pairs...")
pref_data = make_preference_pairs(apps_clean.select(range(min(5, len(apps_clean)))), model, tokenizer, K=3)
print(f"Total preference pairs collected: {len(pref_data)}")
if len(pref_data) > 0:
    print(f"Sample Chosen:\n{pref_data[0]['chosen'][:100]}...")
    print(f"Sample Rejected:\n{pref_data[0]['rejected'][:100]}...")